# Informe EDA: Accidentes y Clima (EE.UU.)
**Integrantes:** [Estudiante 1] · [Estudiante 2] · [Estudiante 3]
**Pregunta de negocio:** ¿qué condiciones (clima, luz, vía, hora) se asocian a accidentes más severos?

> Ejecute las celdas **en orden** (Shift + Enter). Las celdas *Tu interpretación* las escriben ustedes.
> Los CSV en español (accidentes_es.csv y clima_es.csv) deben estar en la misma carpeta que este notebook.

## 0. Preparación: cargar y unir los datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)

acc = pd.read_csv("accidentes_es.csv")
clima = pd.read_csv("clima_es.csv")

# Clima repite hora_inicio, ciudad y estado: las quitamos para no duplicarlas al unir
clima = clima.drop(columns=["hora_inicio", "ciudad", "estado"])

# Unión 1 a 1 por ID (validate avisa si la relación no fuera 1 a 1)
df = acc.merge(clima, on="id", how="inner", validate="one_to_one")
print("Filas y columnas:", df.shape)   # esperado: 100000 filas

In [ ]:
# Variables nuevas a partir de la fecha (las usaremos en los gráficos)
df["hora_inicio"] = pd.to_datetime(df["hora_inicio"])
df["hora"] = df["hora_inicio"].dt.hour
dias = {"Monday":"lunes","Tuesday":"martes","Wednesday":"miércoles","Thursday":"jueves","Friday":"viernes","Saturday":"sábado","Sunday":"domingo"}
df["dia_semana"] = df["hora_inicio"].dt.day_name().map(dias)
df["mes"] = df["hora_inicio"].dt.month
df["severo"] = (df["severidad"] >= 3).astype(int)   # 1 = severidad alta (3 o 4)
df[["hora_inicio", "hora", "dia_semana", "severidad", "severo"]].head()

## 1. Portada y contexto
Escriban aquí: integrantes, pregunta de negocio y 3 objetivos del análisis (use las 3 preguntas que formularon).

## 2. Calidad de datos (herencia de las sesiones 3 y 4)

In [ ]:
print("Duplicados exactos:", df.duplicated().sum())
nulos = df.isna().sum()
print("Columnas con nulos:")
print(nulos[nulos > 0].sort_values(ascending=False))

**Tu interpretación:** resuman en 5 líneas lo ya hecho en el informe consolidado (secciones 6, 7 y 8): qué nulos había, cómo se clasificaron (MAR) y qué outliers se corrigieron.

## 3. Análisis univariado
### 3.1 Tabla descriptiva: media, mediana y asimetría
Si la **media** y la **mediana** se separan mucho y el *skew* es alto, la variable tiene cola larga: reporten la **mediana**.

In [ ]:
num = ["temperatura_f", "humedad_pct", "presion_in", "visibilidad_mi",
       "velocidad_viento_mph", "distancia_mi"]
tabla = df[num].agg(["mean", "median", "std", "min", "max"]).T
tabla["skew"] = df[num].skew()
tabla.round(2)

**Tu interpretación:** ¿cuáles son las 2 variables más asimétricas? ¿qué significa para reportar su centro?

### 3.2 Outliers cuantificados con IQR (no solo 'se ven raros')

In [ ]:
def resumen_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((serie < lim_inf) | (serie > lim_sup)).sum()
    return pd.Series({"Q1": q1, "Q3": q3, "IQR": iqr,
                      "lim_inf": lim_inf, "lim_sup": lim_sup, "outliers": n})

pd.DataFrame({c: resumen_iqr(df[c].dropna()) for c in num}).T.round(2)

**Tu interpretación:** ¿qué variable tiene más outliers? ¿son errores o valores reales? (recuerden las decisiones de la sección 7 del informe).

### 3.3 Gráfico 1: distribución de la severidad

In [ ]:
conteo = df["severidad"].value_counts().sort_index()
ax = conteo.plot(kind="bar", color="#2E74B5")
ax.set_title("La severidad se concentra en los niveles 2 y 3")
ax.set_xlabel("Nivel de severidad")
ax.set_ylabel("Número de accidentes")
plt.xticks(rotation=0)
plt.show()
print((conteo / len(df) * 100).round(1))

**Tu interpretación:** ¿qué porcentaje es severidad 2 y cuál 3? ¿qué pasa con los niveles 1 y 4?

### 3.4 Gráfico 2: histograma de la temperatura (con media y mediana)

In [ ]:
t = df["temperatura_f"].dropna()
plt.hist(t, bins=30, color="#2E74B5", edgecolor="white")
plt.axvline(t.mean(), color="red", label=f"Media {t.mean():.1f}")
plt.axvline(t.median(), color="green", label=f"Mediana {t.median():.1f}")
plt.title("Temperatura en el momento del accidente")
plt.xlabel("Temperatura (°F)")
plt.ylabel("Número de accidentes")
plt.legend()
plt.show()

**Tu interpretación:** ¿la distribución es simétrica? ¿media y mediana coinciden?

### 3.5 Gráfico 3: boxplot de la visibilidad

In [ ]:
plt.boxplot(df["visibilidad_mi"].dropna(), vert=False)
plt.title("Visibilidad: casi todo concentrado en el valor máximo")
plt.xlabel("Visibilidad (millas)")
plt.show()
print(df["visibilidad_mi"].describe().round(2))

**Tu interpretación:** ¿dónde está la caja? ¿por qué hay tantos puntos 'outlier' si la mayoría de los datos vale lo mismo?

### 3.6 Gráfico 4: condiciones climáticas más frecuentes

In [ ]:
top = df["condicion_climatica"].value_counts().head(10).sort_values()
ax = top.plot(kind="barh", color="#2E74B5")
ax.set_title("Condiciones climáticas más frecuentes en los accidentes")
ax.set_xlabel("Número de accidentes")
ax.set_ylabel("")
plt.show()

**Tu interpretación:** ¿qué condición domina? ¿significa que ese clima es el más peligroso? (pista: ¿y cuánto tiempo hace ese clima en la realidad?)

## 4. Análisis bivariado y multivariado

### 4.1 Severidad según la visibilidad
Comparamos la **mediana** de visibilidad por nivel de severidad.

In [ ]:
print(df.groupby("severidad")["visibilidad_mi"].agg(["count", "median", "mean"]).round(2))

df.boxplot(column="visibilidad_mi", by="severidad")
plt.suptitle("")
plt.title("Visibilidad por nivel de severidad")
plt.xlabel("Nivel de severidad")
plt.ylabel("Visibilidad (millas)")
plt.show()

**Tu interpretación:** ¿cambia la visibilidad entre niveles de severidad? Ojo con los niveles que tienen pocos casos.

### 4.2 Tasa de severidad alta por condición climática
`severo` vale 1 si `severidad` ≥ 3. Su promedio es el **porcentaje** de accidentes severos. Solo se usan condiciones con al menos 500 accidentes para que la tasa sea confiable.

In [ ]:
g = df.groupby("condicion_climatica")["severo"].agg(["count", "mean"])
g = g[g["count"] >= 500].sort_values("mean", ascending=False)
g["mean"] = (g["mean"] * 100).round(1)
g = g.rename(columns={"count": "accidentes", "mean": "% severos"})
print("Referencia global: %.1f%% severos" % (df["severo"].mean() * 100))
g

In [ ]:
g["% severos"].sort_values().plot(kind="barh", color="#2E74B5")
plt.axvline(df["severo"].mean() * 100, color="red", linestyle="--", label="Promedio global")
plt.title("Porcentaje de accidentes severos según la condición climática")
plt.xlabel("% de accidentes con severidad 3 o 4")
plt.ylabel("")
plt.legend()
plt.show()

**Tu interpretación:** ¿qué condiciones están por encima del promedio global? ¿la diferencia es grande o pequeña?

### 4.3 Día contra noche

In [ ]:
tabla = pd.crosstab(df["dia_noche"], df["severidad"], normalize="index") * 100
print(tabla.round(1))
(df.groupby("dia_noche")["severo"].mean() * 100).round(1)

**Tu interpretación:** ¿hay diferencia entre día y noche? Si es pequeña, dígalo: 'no hay relación' también es un hallazgo.

### 4.4 Accidentes y severidad por hora del día

In [ ]:
por_hora = df.groupby("hora").agg(accidentes=("id", "count"), pct_severo=("severo", "mean"))
por_hora["pct_severo"] = (por_hora["pct_severo"] * 100).round(1)

fig, ax = plt.subplots()
ax.bar(por_hora.index, por_hora["accidentes"], color="#2E74B5")
ax.set_title("Accidentes por hora del día")
ax.set_xlabel("Hora (0-23)")
ax.set_ylabel("Número de accidentes")
plt.show()
por_hora

**Tu interpretación:** ¿en qué horas se concentran los accidentes? ¿coinciden con las horas de mayor severidad?

### 4.5 Gráfico de dispersión: temperatura contra humedad

In [ ]:
muestra = df.sample(5000, random_state=42)   # muestra para que el gráfico sea legible
plt.scatter(muestra["temperatura_f"], muestra["humedad_pct"], s=5, alpha=0.4)
plt.title("A mayor temperatura, menor humedad")
plt.xlabel("Temperatura (°F)")
plt.ylabel("Humedad (%)")
plt.show()
print("Pearson:", round(df["temperatura_f"].corr(df["humedad_pct"]), 2))

**Tu interpretación:** ¿qué dirección y fuerza tiene la relación? Recuerden la tabla: |r| de 0,3 a 0,5 es moderada y mayor a 0,7 es muy fuerte.

### 4.6 Mapa de calor de correlaciones (multivariado)

In [ ]:
cols = ["severidad", "temperatura_f", "humedad_pct", "presion_in",
        "visibilidad_mi", "velocidad_viento_mph", "distancia_mi", "hora"]
corr = df[cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im)
ax.set_title("Correlaciones de Pearson entre variables numéricas")
plt.tight_layout()
plt.show()

**Tu interpretación:** ¿qué par de variables tiene la correlación más fuerte? ¿alguna con `severidad` es fuerte? Recuerden: correlación no es causalidad.

### 4.7 Elementos de la vía y severidad
Para cada elemento (semáforo, intersección...) comparamos el % de accidentes severos **con** y **sin** él.

In [ ]:
vias = ["semaforo", "interseccion", "cruce_peatonal", "senal_pare", "ferrocarril"]
filas = []
for v in vias:
    g = df.groupby(v)["severo"].agg(["count", "mean"])
    filas.append({"elemento": v,
                  "accidentes_con": g.loc[True, "count"],
                  "% severos con": round(g.loc[True, "mean"] * 100, 1),
                  "% severos sin": round(g.loc[False, "mean"] * 100, 1)})
pd.DataFrame(filas)

**Tu interpretación:** ¿qué elementos se asocian a menos severidad y cuáles a más? ¿puede ser que la velocidad de la vía sea la causa de fondo? (asociación no es causalidad).

## 5. Hallazgos y recomendaciones
Escriban **mínimo 3** con este formato (para lectores no técnicos):

| Hallazgo | Evidencia (cifra o gráfico) | Acción sugerida |
|---|---|---|
| | | |
| | | |
| | | |

## 6. Anexo
Este notebook, con todo el código ejecutado, es el anexo reproducible.